<a href="https://colab.research.google.com/github/DebStar17/GNSS-Anti-spoofing/blob/main/Hackathon_GNSS_anti_spoofing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# CONFIGURATION (Change file names here)
# ==========================================
TRAIN_FILE = 'train.csv'
TEST_FILE = 'test.csv'
SUBMISSION_TEMPLATE = 'submission_format.csv'
OUTPUT_FILE = 'final_submission.csv'

# The 5 independent physics-based features we defined
FEATURES = ['Phase_Divergence', 'Doppler_Delta', 'Corr_Symmetry', 'CN0_Stability', 'Tracking_Error']

# ==========================================
# 1. CORE ENGINE: CLEANUP & PHYSICS MATH
# ==========================================
def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading and cleaning {file_path}...")
    df = pd.read_csv(file_path, low_memory=False)

    # GARBAGE CLEANUP: The first few rows have strings like 'ch0' in numeric columns.
    # This forces 'Carrier_Doppler_hz' to be a number. If it's garbage ('ch0'), it turns into NaN.
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy() # Drop the garbage rows

    # Convert all necessary columns to floats so math works
    cols_to_float = ['Carrier_Doppler_hz', 'Pseudorange_m', 'Carrier_phase',
                     'EC', 'LC', 'PC', 'PIP', 'PQP', 'CN0', 'time']
    for col in cols_to_float:
        df[col] = df[col].astype(float)

    if is_train:
        df['spoofed'] = df['spoofed'].astype(int)

    # Sort by channel then time so our 'Delta' and 'Rolling' math is chronologically correct
    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    print("--> Calculating physically independent features...")
    # Feature 1: Carrier-Phase Divergence (lambda approx 0.19m for L1)
    df['Phase_Divergence'] = df['Pseudorange_m'] - (0.19 * df['Carrier_phase'])

    # Feature 2: Doppler Rate-of-Change (Kinematic acceleration)
    df['Doppler_Delta'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().fillna(0)

    # Feature 3: Correlation Symmetry (Peak distortion)
    epsilon = 1e-9 # Prevents division by zero
    df['Corr_Symmetry'] = (df['EC'] - df['LC']) / (df['EC'] + df['LC'] + epsilon)

    # Feature 4: CN0 Stability (Jitter/Noise rolling standard deviation)
    df['CN0_Stability'] = df.groupby('channel')['CN0'].transform(lambda x: x.rolling(window=5, min_periods=1).std().fillna(0))

    # Feature 5: Tracking Loop Error (Phase Lock deviation)
    df['Tracking_Error'] = np.arctan2(df['PQP'], df['PIP'])

    return df

# ==========================================
# 2. PHASE 1: TRAINING (Finding the t's and c's)
# ==========================================
print("\n--- STARTING TRAINING PHASE ---")
train_df = load_and_engineer(TRAIN_FILE, is_train=True)

X_train = train_df[FEATURES]
y_train = train_df['spoofed']

# The Random Forest acts as our automated optimizer.
# It finds the thresholds (t) and assigns importance weights (c) internally.
print("--> Training model to determine optimal thresholds and weights...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Show the 'c' values (Feature Importances) to prove the physics works
print("\n[Model Weights / Importance (c)]:")
for name, importance in zip(FEATURES, rf_model.feature_importances_):
    print(f" - {name}: {importance:.4f}")


--- STARTING TRAINING PHASE ---
--> Loading and cleaning train.csv...
--> Calculating physically independent features...
--> Training model to determine optimal thresholds and weights...

[Model Weights / Importance (c)]:
 - Phase_Divergence: 0.9159
 - Doppler_Delta: 0.0233
 - Corr_Symmetry: 0.0191
 - CN0_Stability: 0.0250
 - Tracking_Error: 0.0166


In [7]:
# ==========================================
# 3. PHASE 2: TESTING & GENERATING SUBMISSION
# ==========================================
print("\n--- STARTING TESTING PHASE ---")
test_df = load_and_engineer(TEST_FILE, is_train=False)

X_test = test_df[FEATURES]

print("--> Predicting confidence scores on test data...")
# rf_model.predict_proba outputs [Probability_Clean, Probability_Spoofed]
# We want the Probability_Spoofed (index 1), which acts as our 'Confidence' score
test_df['Confidence'] = rf_model.predict_proba(X_test)[:, 1]

# IMPORTANT HACKATHON LOGIC:
# The data gives us metrics per 'channel' (satellite), but the submission format
# requires ONE verdict per 'time' step.
# Logic: If ANY channel looks highly spoofed at time T, the whole receiver is spoofed at time T.
print("--> Aggregating channel scores into final receiver verdict...")
submission_agg = test_df.groupby('time')['Confidence'].max().reset_index()

# If the confidence is > 50%, we declare it Spoofed (1). Otherwise, Genuine (0).
submission_agg['Spoofed'] = (submission_agg['Confidence'] > 0.5).astype(int)

# Format to match submission_format.csv exactly
submission_final = submission_agg[['time', 'Spoofed', 'Confidence']]
submission_final['time'] = submission_final['time'].astype(int)

# Save the file
submission_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nSUCCESS! Submission saved to '{OUTPUT_FILE}'. You are ready to upload.")


--- STARTING TESTING PHASE ---
--> Loading and cleaning test.csv...
--> Calculating physically independent features...
--> Predicting confidence scores on test data...
--> Aggregating channel scores into final receiver verdict...

SUCCESS! Submission saved to 'final_submission.csv'. You are ready to upload.
